In [1]:
from instances.instances import generate_instances

instances = generate_instances(filename="TSP50.pkl", instance_count=1000, cities=50, seed=42)

In [2]:
from data.generation import generate_train_data
from data.adapters.input.sparse import SparseInputAdapter
from data.adapters.output.basic import BasicOutputAdapter

input_config = (SparseInputAdapter, 50)
output_config = (BasicOutputAdapter, 50)

generate_train_data(
    instance_file="TSP50.pkl", 
    data_filename="train_data.h5", 
    input_adapter_config=input_config, 
    output_adapter_config=output_config,
    num_workers=12,
    size=10000
)

Datos guardados en: /home/oscar/Escritorio/TSP-Framework/data/train_data.h5 (Tamaño: 10000)


In [1]:
from data.preprocessing import load_dataset, split_dataset

train_file, val_file = split_dataset("train_data.h5", 8000)

train_dataset = load_dataset(train_file)
val_dataset = load_dataset(val_file)

Cargando dataset original: /home/oscar/Escritorio/TSP-Framework/data/train_data.h5
Guardando Train puro (8000 muestras) en: train_data_train.h5
Guardando Test puro (2000 muestras) en: train_data_test.h5
Dataset train_data_train.h5 cargado con 8000 muestras.
Dataset train_data_test.h5 cargado con 2000 muestras.


In [2]:
from models.sparse import TSPTransformer

# Parámetros del modelo
input_dim = 2
embed_dim = 64
num_heads = 4
num_encoder_layers = 2
num_glimpses = 2
dropout = 0.1

# Crear modelo
model = TSPTransformer(
    input_dim=input_dim,
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_encoder_layers=num_encoder_layers,
    num_glimpses=num_glimpses,
    dropout_rate=dropout
)

In [3]:
from training.sl import train, LRConfig
from training.metrics import CrossEntropyLoss, Accuracy

model = train(
    model=model,
    epochs=10,
    train_set=train_dataset,
    val_set=val_dataset,
    batch_size=64,
    lr_config=LRConfig(value=1e-4),
    weight_decay=1e-5,
    loss_fn=CrossEntropyLoss(),
    patience=10,
    metrics=[Accuracy()]
)

ℹ️ Usando dispositivo: cpu

Epoch 1/10
    Train CrossEntropy: 1.9127 | Val CrossEntropy: 1.2202
    Accuracy: 65.35%
Epoch 2/10
    Train CrossEntropy: 1.0946 | Val CrossEntropy: 0.8604
    Accuracy: 76.05%
Epoch 3/10
    Train CrossEntropy: 0.9095 | Val CrossEntropy: 0.7554
    Accuracy: 78.50%
Epoch 4/10
    Train CrossEntropy: 0.8351 | Val CrossEntropy: 0.6993
    Accuracy: 79.80%
Epoch 5/10
    Train CrossEntropy: 0.7790 | Val CrossEntropy: 0.6685
    Accuracy: 79.40%
Epoch 6/10
    Train CrossEntropy: 0.7422 | Val CrossEntropy: 0.6327
    Accuracy: 81.00%
Epoch 7/10
    Train CrossEntropy: 0.7207 | Val CrossEntropy: 0.6264
    Accuracy: 80.05%
Epoch 8/10
    Train CrossEntropy: 0.6916 | Val CrossEntropy: 0.5965
    Accuracy: 80.80%
Epoch 9/10
    Train CrossEntropy: 0.6765 | Val CrossEntropy: 0.5916
    Accuracy: 81.05%
Epoch 10/10
    Train CrossEntropy: 0.6618 | Val CrossEntropy: 0.5767
    Accuracy: 81.55%

** Mejor modelo restaurado (Época 10): CrossEntropy = 0.5767
